# grad-accumulate-on-leaf — worked example 1: Two-path leaf gradient accumulation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-accumulate-on-leaf`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When a leaf tensor appears multiple times in the computation graph, its gradient is the SUM of contributions from every path that leads back to it. The accumulation rule is: set `leaf.grad = g` on the first backward visit (when `leaf.grad is None`), and `leaf.grad = leaf.grad + g` on every subsequent visit. This is why `optimizer.zero_grad()` must be called between training steps — leftover gradients from the previous step would corrupt the next update.

## Worked solution

**Step 1 — create a leaf parameter.** We create `w` as a scalar leaf with `requires_grad=True`. Its `.grad` starts as `None`.

**Step 2 — build a two-path graph.** The expression `loss = w * w + 3 * w` means `w` appears in two separate sub-expressions. When we call `loss.backward()`, PyTorch will visit `w` once from each path.

**Step 3 — verify accumulation.** After `backward()`, `w.grad` should equal the analytical derivative of `w**2 + 3*w`, which is `2*w + 3`. At `w=4.0` that is `2*4 + 3 = 11.0`. If PyTorch only kept the last path's contribution, we'd get a wrong answer.

**Step 4 — demonstrate zero_grad.** After zeroing, `w.grad` is `None` again. A second forward+backward now produces a FRESH gradient of `11.0`, not `22.0` (which would happen if gradients accumulated across steps).

In [ ]:
import torch as t

t.manual_seed(0)

# A scalar leaf parameter
w = t.tensor(4.0, requires_grad=True)

# Two-path computation: w appears in both terms
loss = w * w + 3.0 * w

# Backward accumulates contributions from both paths
loss.backward()

# Analytical gradient: d/dw (w^2 + 3w) = 2w + 3 = 11 at w=4
print(f"w.grad after backward: {w.grad.item()}")  # expect 11.0
assert abs(w.grad.item() - 11.0) < 1e-5

# zero_grad resets the slate
w.grad = None
print(f"w.grad after zero_grad: {w.grad}")  # expect None

# Second step: fresh gradient (NOT accumulated from previous step)
loss2 = w * w + 3.0 * w
loss2.backward()
print(f"w.grad after second backward (no accumulation): {w.grad.item()}")  # expect 11.0
assert abs(w.grad.item() - 11.0) < 1e-5
print("All checks passed.")